<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Feynman_Path_Integral_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualization of the Feynman Path Integral

This project provides an interactive and high-fidelity visualization of Richard Feynman's Path Integral formulation, demonstrating the fundamental quantum mechanical transition from a 'sum over all possible histories' to the emergence of classical deterministic motion. By simulating 400 distinct trajectories using randomized harmonics and animating them sequentially, the notebook illustrates how non-classical paths interfere destructively while the stationary action path (the classical trajectory) emerges through constructive interference, offering a clear visual bridge between quantum and classical physics.

## Theoretical Background

### The Principle of Stationary Action
In classical mechanics, a particle moving from Point A to Point B follows a single unique trajectory: the path that minimizes the action functional $S = \int L dt$. This is known as the Principle of Least Action.

### The Quantum Mechanical Formulation
Richard Feynman suggested that a particle simultaneously explores every conceivable path. Each path contributes a complex probability amplitude: $\phi(\text{path}) \propto e^{iS/\hbar}$.

### Constructive and Destructive Interference
*   **Non-Classical Paths**: These oscillate rapidly and interfere destructively, canceling each other out.
*   **The Classical Path**: Near the stationary action, paths interfere constructively, forming the dominant trajectory.

## Code Overview
This implementation is divided into four main functional blocks:

1.  **Configuration**: Defines physical boundaries (Points A & B), the number of simulated histories (400), and the timing for the sequential animation reveal.
2.  **Path Generation**: Uses randomized sine-wave harmonics to create deviations from the classical straight-line path. A 'centrality' factor ensures that paths closer to the classical center have lower maximum amplitudes of deviation.
3.  **Animation Engine**: Utilizes `matplotlib.animation.FuncAnimation` to iterate through the path list. It handles the 'Ghostly' effect by dynamically updating the alpha (transparency) and color of lines as they transition from the active 'exploration' state to the background 'history' state.
4.  **Export Utility**: A dedicated cell that resets the animation state and uses `FFMpeg` to encode the frames into a high-bitrate MP4 file for external use.

## Technical Implementation
This notebook simulates a "sum over histories" using 400 randomized harmonic paths.

### Key Visual Logic
- **Ghostly Paths**: Non-classical trajectories are rendered as semi-transparent blue trails that appear one by one. This sequential reveal demonstrates the exhaustive nature of the path summation.
- **Classical Path**: As the paths aggregate, the stationary action path emerges via constructive interference, highlighted at the end of the sequence.
- **Timing & Frames**: The animation utilizes a sequential exploration phase (400 frames) followed by a final 90-frame reveal (approx. 3s) to solidify the classical trajectory.
- **Requirements**: Built using `numpy`, `matplotlib` (FuncAnimation), and `IPython.display`.

In [ ]:
"""
Feynman Path Integral Visualization
===================================
Author: Mugambi Ndwiga
Social: @craftsandengineering

This script generates a high-density animation of the Feynman Path Integral,
visualizing the Principle of Stationary Action. It simulates 400 possible
paths (histories) from Point A to Point B using random harmonics.

Key Visual Logic:
- Ghostly paths: Non-classical paths that appear one by one and interfere destructively.
- Classical path: The path of stationary action that emerges via constructive interference.
- Timing: Sequential exploration of paths followed by a final reveal.

Requirements: numpy, matplotlib, IPython
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import hsv_to_rgb
from IPython.display import HTML, display
import matplotlib

# Increase memory limit for dense animation embedding
matplotlib.rcParams['animation.embed_limit'] = 150.0

# --- Configuration and Timing Parameters ---
n_paths = 400
points_per_path = 50

# Timing logic
frames_per_path = 1
total_sequential_frames = n_paths * frames_per_path
final_pause_frames = 90
total_frames = total_sequential_frames + final_pause_frames

# Boundary coordinates for Points A and B
x_start, y_start = -2, 0
x_end, y_end = 2, 0

# --- Plot Setup ---
fig, ax = plt.subplots(figsize=(12, 8), facecolor='black')
ax.set_facecolor('black')
ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-1.8, 1.8)
ax.axis('off')

# Visual elements: Title, Information Box, and Credits
title_text = ax.text(0.5, 0.95, "Feynman Path Integral: Principle of Stationary Action",
                     transform=ax.transAxes, color='white', fontsize=18,
                     fontweight='bold', ha='center', va='bottom')

info_box = ax.text(0.5, 0.05, "", transform=ax.transAxes, color='cyan',
                   fontsize=12, ha='center', va='top', style='italic')

# Probable path annotation (hidden initially)
classical_label = ax.text(0, 0.2, "Most Probable Path (Classical)",
                          color='white', fontsize=12, fontweight='bold',
                          ha='center', va='bottom', alpha=0)

# Professional watermark
ax.text(0.99, 0.01, '@craftsandengineering | Mugambi Ndwiga', transform=ax.transAxes,
        color='white', alpha=0.3, ha='right', va='bottom', fontsize=8)

# Initialize path lines with 0 alpha
paths = []
for _ in range(n_paths):
    line, = ax.plot([], [], lw=0.6, alpha=0)
    paths.append(line)

# Marker setup for Source (A) and Destination (B)
ax.plot(x_start, y_start, 'yo', markersize=10, zorder=10)
ax.plot(x_end, y_end, 'ro', markersize=10, zorder=10)
ax.text(x_start, y_start - 0.15, "Point A", color='yellow', ha='center', fontsize=10)
ax.text(x_end, y_end - 0.15, "Point B", color='red', ha='center', fontsize=10)

# --- Data Generation ---
# Pre-calculating path coordinates using randomized harmonics (Sine waves)
x_coords = np.linspace(x_start, x_end, points_per_path)
path_data = []

for i in range(n_paths):
    deviation = np.zeros_like(x_coords)
    # Centrality determines how far the path strays from the classical straight line
    centrality = (i - n_paths/2) / (n_paths/2)
    n_harmonics = np.random.randint(1, 5)
    for h in range(1, n_harmonics + 1):
        amp = np.random.uniform(-0.6, 0.6) * (centrality**2)
        deviation += amp * np.sin(h * np.pi * (x_coords - x_start) / (x_end - x_start))
    path_data.append(deviation)

def update(frame):
    """Update function for the Matplotlib animation loop to reveal paths one by one."""
    current_path_idx = frame // frames_per_path

    if frame < total_sequential_frames:
        info_box.set_text(f"Summing over all possible histories... Path {min(current_path_idx + 1, n_paths)}")

        # Reveal only the current path appearing this frame
        line = paths[current_path_idx]
        line.set_data(x_coords, path_data[current_path_idx])
        line.set_color('white')
        line.set_alpha(0.8)
        line.set_linewidth(0.8)

        # Set previous path to ghostly state
        if current_path_idx > 0:
            prev_line = paths[current_path_idx - 1]
            prev_line.set_alpha(0.15)
            prev_line.set_linewidth(0.4)
            prev_line.set_color(hsv_to_rgb((0.6, 0.4, 0.8)))

        classical_label.set_alpha(0)
    else:
        # Final phase: Stationary action emerges naturally from density
        info_box.set_text("Stationary action emerges from the sum of all paths.")
        # Ensure the very last sequential path is also set to ghostly
        paths[n_paths-1].set_alpha(0.15)
        paths[n_paths-1].set_color(hsv_to_rgb((0.6, 0.4, 0.8)))
        # Fade in the annotation at the end
        classical_label.set_alpha(1)

    return paths + [info_box, classical_label]

# Create and display animation
ani = FuncAnimation(fig, update, frames=total_frames, blit=True, interval=25)

plt.close()
display(HTML(ani.to_jshtml()))

print("Animation rendering complete.")

In [3]:
"""
Video Export Utility
--------------------
Encodes the generated animation into an MP4 file and triggers a browser download.
Requires FFmpeg to be installed on the system.
"""
from google.colab import files
from matplotlib.animation import FFMpegWriter

print("Resetting animation state for clean export...")
# Ensure all paths are hidden before the video encoder starts
for line in paths:
    line.set_data([], [])
    line.set_alpha(0)

print("Encoding animation to MP4 using FFmpeg...")

# Configure the writer for H.264 high-quality output
writer = FFMpegWriter(fps=30, metadata=dict(artist='Mugambi Ndwiga'), bitrate=1800)

# Save the animation to the local Colab disk
# bit_rate and fps are matched to the visual complexity
ani.save("feynman_path_integral.mp4", writer=writer)

print("Saving complete. Initiating browser download...")

# Trigger browser download prompt
files.download('feynman_path_integral.mp4')

Resetting animation state for clean export...
Encoding animation to MP4 using FFmpeg...
Saving complete. Initiating browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>